In [2]:
import os
import pandas as pd
from scipy.stats import gaussian_kde

In [3]:
DATA_DIR = '/home/mkeoliya/projects/CAREBench/data/mcmed'

DEMOGRAPHICS

In [32]:
ETHNICITY = ['Non-Hispanic/Non-Latino', 'Hispanic/Latino']

In [33]:
RACES = ['Other', 'White', 'Black or African American', 'Asian',
         'Native Hawaiian or Other Pacific Islander', 'American Indian or Alaska Native']

In [34]:
demo_df = pd.read_csv(os.path.join(DATA_DIR, 'visits.csv'), usecols=['CSN', 'Age', 'Gender', 'Race', 'Ethnicity', 'Arrival_time'])

In [35]:
demo_df['Ethnicity'] = demo_df['Ethnicity'].apply(lambda x: x if x in ETHNICITY else 'Unknown')
demo_df['Race'] = demo_df['Race'].apply(lambda x: x if x in RACES else 'Unknown')

In [36]:
demo_df['Arrival_time'] = pd.to_datetime(demo_df['Arrival_time'].apply(lambda x: str(int(x[:4]) - 500) + x[4:]))

In [ ]:
demo_df.to_csv(os.path.join(DATA_DIR, 'demo.csv'), index=False)

NUMERICS

In [55]:
triage_df = pd.read_csv(os.path.join(DATA_DIR, 'visits.csv'), 
    usecols=['CSN', 'Triage_Temp', 'Triage_HR','Triage_RR', 'Triage_SpO2', 'Triage_SBP', 'Triage_DBP', 'Arrival_time'])
triage_df = triage_df.rename(columns={'Triage_Temp': 'Temp', 'Triage_HR': 'HR', 'Triage_RR': 'RR', 'Triage_SpO2': 'SpO2', 'Triage_SBP': 'SBP', 'Triage_DBP': 'DBP', 'Arrival_time': 'Time'})

In [56]:
triage_df = pd.melt(
    triage_df,
    id_vars=['CSN', 'Time'],    # columns to keep as identifiers
    var_name='Measure',         # new column for former column names
    value_name='Value'          # new column for the data values
)

In [53]:
triage_df = triage_df.dropna()

In [10]:
numerics_df = pd.read_csv(os.path.join(DATA_DIR, 'numerics.csv'), usecols=['CSN', 'Time', 'Measure', 'Value'])

In [11]:
numerics_df = pd.concat([triage_df, numerics_df], ignore_index=True).sort_values(by=['CSN', 'Time'])

In [12]:
numerics_df['Time'] = pd.to_datetime(numerics_df['Time'].apply(lambda x: str(int(x[:4]) - 500) + x[4:]))

In [ ]:
numerics_df.to_csv(os.path.join(DATA_DIR, 'numerics.csv'), index=False)

LABS

In [14]:
labs_df = pd.read_csv(os.path.join(DATA_DIR, 'labs.csv'), usecols=['CSN', 'Order_time', 'Result_time', 'Component_name', 'Component_value', 'Component_nml_low', 'Component_nml_high'])

In [15]:
labs_df = labs_df.rename(columns={'Order_time': 'Time', 'Component_name': 'Measure', 'Component_value': 'Value', 'Component_nml_low': 'Low', 'Component_nml_high': 'High'})

In [ ]:
labs_df['Time'] = pd.to_datetime(labs_df['Time'].apply(lambda x: str(int(x[:4]) - 500) + x[4:]))
labs_df['Result_time'] = pd.to_datetime(labs_df['Result_time'].apply(lambda x: str(int(x[:4]) - 500) + x[4:]))

In [ ]:
labs_df.to_csv(os.path.join(DATA_DIR, 'labs.csv'), index=False)

ORDERS

In [18]:
orders_df = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'), usecols=['CSN', 'Order_time', 'First_admin_time', 'Result_time', 'Procedure_name'])

In [19]:
orders_df['Result_time'] = orders_df['Result_time'].fillna(orders_df['First_admin_time'])
orders_df = orders_df.drop(columns=['First_admin_time'])

In [20]:
orders_df = orders_df.rename(columns={'Order_time': 'Time', 'Procedure_name': 'Measure'})

In [ ]:
orders_df['Time'] = pd.to_datetime(orders_df['Time'].apply(lambda x: str(int(x[:4]) - 500) + x[4:]))
orders_df['Result_time'] = pd.to_datetime(orders_df['Result_time'].apply(lambda x: str(int(x[:4]) - 500) + x[4:]))

In [ ]:
orders_df.to_csv(os.path.join(DATA_DIR, 'orders.csv'), index=False)

SPLIT

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'split_random_train.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'split_random_val.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'split_random_test.csv'))

In [ ]:
train_df['split'] = 'train'
train_df['split'] = 'train'
train_df['split'] = 'train'

In [ ]:
split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

In [ ]:
split_df.to_csv(os.path.join(DATA_DIR, 'split.csv'), index=False)

Add TIME

In [24]:
def add_time_to_arrival(name, demo_df):
    df = pd.read_csv(os.path.join(DATA_DIR, f'{name}.csv'))
    df = df.merge(demo_df, on='CSN')
    df['Arrival_time'] = pd.to_datetime(df['Arrival_time'])
    df['Time'] = pd.to_datetime(df['Time'])
    df['time_arrive'] = (df['Time'] - df['Arrival_time']).dt.total_seconds() / 3600
    df = df.drop(columns=['Arrival_time'])
    df.to_csv(os.path.join(DATA_DIR, f'{name}.csv'), index=False)

In [8]:
demo_df = pd.read_csv(os.path.join(DATA_DIR, 'demo.csv'), usecols=['CSN', 'Arrival_time'])

In [25]:
add_time_to_arrival('numerics', demo_df)
add_time_to_arrival('labs', demo_df)
add_time_to_arrival('orders', demo_df)

In [ ]:
def add_time_to_result(fname, demo_df):
    df = pd.read_csv(os.path.join(DATA_DIR, f'{fname}.csv'))
    df = df.merge(demo_df[['CSN', 'Arrival_time']], on='CSN')

    df['Result_time'] = df.apply(
        lambda row: row['Time'] 
                    if pd.isna(row['Result_time']) 
                    else str(int(row['Result_time'][:4]) - 500) + row['Result_time'][4:], 
        axis=1
    )

    df['Result_time'] = pd.to_datetime(df['Result_time'], format='mixed', utc=True)
    df['Arrival_time'] = pd.to_datetime(df['Arrival_time'])
    df['time_result'] = (df['Result_time'] - df['Arrival_time']).dt.total_seconds() / 3600
    df = df.drop(columns=['Arrival_time'])

df.to_csv(os.path.join(DATA_DIR, f'{fname}.csv'), index=False)

In [ ]:
add_time_to_result('labs', demo_df)
add_time_to_result('orders', demo_df)

TEXT

In [ ]:
rads_df = pd.read_csv(os.path.join(DATA_DIR, 'rads.csv'))
rads_df = rads_df[rads_df['CSN'].isin(label_df['CSN'])]

In [ ]:
rads_df = rads_df.merge(visits_df[['CSN', 'Arrival_time']], on='CSN')

In [ ]:
rads_df['Order_time'] = pd.to_datetime(rads_df['Order_time'].apply(lambda x: str(int(x[:4]) - 500) + x[4:]))
rads_df['Result_time'] = pd.to_datetime(rads_df['Result_time'].apply(lambda x: str(int(x[:4]) - 500) + x[4:]))
rads_df['Arrival_time'] = pd.to_datetime(rads_df['Arrival_time'].apply(lambda x: str(int(x[:4]) - 500) + x[4:]))

In [ ]:
rads_df['Order_time'] = rads_df['Order_time'] - rads_df['Arrival_time']
rads_df['Result_time'] = rads_df['Result_time'] - rads_df['Arrival_time']
rads_df = rads_df.drop(columns=['Arrival_time'])

In [ ]:
rads_df['Order_time'] = rads_df['Order_time'].dt.total_seconds() / 3600
rads_df['Result_time'] = rads_df['Result_time'].dt.total_seconds() / 3600

In [ ]:
rads_df = rads_df.merge(label_df[['CSN', 'Time']], on='CSN')
rads_df = rads_df[rads_df['Order_time'] <= (rads_df['Time'])]
rads_df = rads_df.drop(columns=['Time'])

In [ ]:
rads_df.to_parquet(f'data/{TASK}_rads.parquet')

WAVEFORMS

In [ ]:
import wfdb
import numpy as np
from scipy.stats import linregress, kurtosis, skew

In [ ]:
def extract_stats(xs):
    xs = np.array(xs)
    x = np.arange(len(xs))
    slope, _, _, _, _ = linregress(x, xs)

    return {
        'min': xs.min(),
        'max': xs.max(),
        'mean': xs.mean(),
        'std': xs.std(),
        'median': np.median(xs),
        'skew': skew(xs),
        'kurtosis': kurtosis(xs),
        'slope': slope,
        'qr_25': np.quantile(xs, 0.25),
        'qr_75': np.quantile(xs, 0.75),
    }

In [ ]:
label_ids = pd.read_csv(os.path.join(DATA_DIR, 'demo.csv'), usecols=['CSN', 'Arrival_time'])

In [ ]:
summary_df = pd.read_csv(os.path.join(DATA_DIR, 'data', 'waveform_summary.csv'))
summary_df = summary_df[(summary_df['CSN'].isin(label_ids)) & (summary_df['Type'] == 'II')]

In [ ]:
waveform_dict = {}
duration_dict = {}
for csn in summary_df['CSN'].unique():
    data_paths = (os.path.join(DATA_DIR, 'waveforms', str(csn)[-3:], str(csn), 'II'))
    arrival_time = visits_df[visits_df['CSN'] == csn].iloc[0].Arrival_time
    arrival_time = arrival_time.replace(tzinfo=None)
    signals = []
    
    for data_path in os.listdir(data_paths):
        if data_path[-3:] == 'dat':
            record_path = os.path.join(data_paths, data_path[:-4])
            record = wfdb.rdrecord(record_path)
            signal = record.p_signal.reshape(-1)
            duration = (len(record.p_signal.reshape(-1)) / 250) / 3600

            base_time = (record.base_datetime - pd.DateOffset(years=500) - arrival_time).total_seconds()
            num_samples = len(signal)
            time = np.linspace(0, num_samples / record.fs, num_samples) + base_time
            # mask = time < N 
            base_time = base_time / 3600
            
            signal = signal[mask]
            signals += list(signal)
    
    if len(signals) > 0:
        waveform_dict[csn] = signals
        duration_dict[csn] = base_time, base_time + duration

In [ ]:
import pickle
with open(f'{TASK}_dict.pkl', 'wb') as f:
    pickle.dump(waveform_dict, f)

DEMO EVENTVAL

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'demo.csv'))

In [47]:
df['Gender_ind'] = pd.factorize(df['Gender'])[0]
df['Race_ind'] = pd.factorize(df['Race'])[0]
df['Ethnicity_ind'] = pd.factorize(df['Ethnicity'])[0]
df['Age_ind'] = df['Age']

In [48]:
df['Race_str'] = 'Race|' + df['Race']
df['Ethnicity_str'] = 'Ethnicity|' + df['Ethnicity']
df['Gender_str'] = 'Gender|' + df['Gender']

In [49]:
bins = [0, 20, 30, 40, 50, 60, 70, 80, 90]
labels = ['0-20', '20-30', '30-40', '40-50', '50-60', '60-70', '70-80', '80-90']
df['Age_str'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True, include_lowest=True)

In [ ]:
df.to_csv(os.path.join(DATA_DIR,'demo.csv'), index=False)

BUCKETS

In [23]:
import numpy as np
import pickle

In [24]:
scale = 10_000  # for 4 decimal places

In [ ]:
def try_parse_float(x):
    if isinstance(x, str):
        try:
            x = x.lstrip('<>=').strip()
            x = x.replace(',', ':')
            if ':' in x:
                try:
                    num, denom = x.split(':')
                    return float(num) / float(denom)
                except (ValueError, ZeroDivisionError):
                    return x
            return float(x)
        except ValueError:
            return x
    return x

In [10]:
labs_df = pd.read_csv(os.path.join(DATA_DIR, 'labs.csv'), usecols=['CSN', 'Measure', 'Value'])

In [ ]:
labs_df['Value'] = labs_df['Value'].replace([None], 0.0).apply(try_parse_float)
labs_df['Value'] = labs_df['Value'].apply(lambda x: 1.0 if isinstance(x, str) and 'pos' in x.lower() else x)
labs_df['Value'] = labs_df['Value'].apply(lambda x: 0.0 if isinstance(x, str) and (any(sub in x.lower() for sub in ['neg', 'not', 'none', 'auto'])) else x)

In [25]:
labs = {}
lab_names = labs_df['Measure'].unique()

for l in lab_names:
    df = labs_df[labs_df['Measure'] == l]
    bucket = np.percentile(df['Value'], np.arange(0, 110, 10))
    bucket = np.array([
            np.floor(bucket[0] * scale) / scale,             # floor first
            *np.round(bucket[1:-1] * scale) / scale,         # round middle
            np.ceil(bucket[-1] * scale) / scale              # ceil last
        ])
    labs[l] = bucket[np.concatenate(([True], bucket[1:] != bucket[:-1]))]

In [ ]:
with open(os.path.join(DATA_DIR, 'labs_buckets.pkl'), 'wb') as f:
    pickle.dump(labs, f)

In [59]:
numerics_df = pd.read_csv(os.path.join(DATA_DIR, 'numerics.csv'), usecols=['CSN', 'Measure', 'Value'])

In [60]:
numerics_df = numerics_df.dropna()

In [63]:
numerics = {}
numerics_names = numerics_df['Measure'].unique()

for n in numerics_names:
    df = numerics_df[numerics_df['Measure'] == n]
    bucket = np.percentile(df['Value'], np.arange(0, 110, 10))
    bucket = np.array([
            np.floor(bucket[0] * scale) / scale,             # floor first
            *np.round(bucket[1:-1] * scale) / scale,         # round middle
            np.ceil(bucket[-1] * scale) / scale              # ceil last
        ])
    numerics[n] = bucket[np.concatenate(([True], bucket[1:] != bucket[:-1]))]

In [ ]:
with open(os.path.join(DATA_DIR, 'numerics_buckets.pkl'), 'wb') as f:
    pickle.dump(numerics, f)

NUMERICS EVENTVAL

In [ ]:
def bucket_eventval(event, val, d):
    buckets = d[event]
    ind = np.searchsorted(buckets, val, side='right')
    if ind == len(buckets):
        eventval = f"{event}|{buckets[ind-1]}-"
    else:
        eventval = f"{event}|{buckets[ind-1]}-{buckets[ind]}"
    return eventval

In [55]:
def bucket_ind(event, val, d):
    buckets = d[event]
    ind = np.searchsorted(buckets, val, side='right')
    return ind

In [ ]:
with open(os.path.join(DATA_DIR, 'numerics_buckets.pkl'), 'rb') as f:
    buckets = pickle.load(f)

In [56]:
numerics_df = pd.read_csv(os.path.join(DATA_DIR, 'numerics.csv'))

In [63]:
numerics_df['eventval'] = numerics_df.apply(lambda x: bucket_eventval(x['Measure'], x['Value'], buckets), axis=1)
numerics_df['buckets'] = numerics_df.apply(lambda x: bucket_ind(x['Measure'], x['Value'], buckets), axis=1)

In [ ]:
numerics_df.to_csv(os.path.join(DATA_DIR, 'numerics.csv'), index=False)

LABS EVENTVAL

In [ ]:
with open(os.path.join(DATA_DIR, 'labs_buckets.pkl'), 'rb') as f:
    buckets = pickle.load(f)

In [ ]:
labs_df = pd.read_csv(os.path.join(DATA_DIR, 'labs.csv'))

In [72]:
labs_df['Value'] = labs_df['Value'].replace([None], 0.0).apply(try_parse_float)
labs_df['Value'] = labs_df['Value'].apply(lambda x: 1.0 if isinstance(x, str) and 'pos' in x.lower() else x)
labs_df['Value'] = labs_df['Value'].apply(lambda x: 0.0 if isinstance(x, str) and (any(sub in x.lower() for sub in ['neg', 'not', 'none', 'auto'])) else x)

In [74]:
labs_df['eventval'] = labs_df.apply(lambda x: bucket_eventval(x['Measure'], x['Value'], buckets), axis=1)
labs_df['buckets'] = labs_df.apply(lambda x: bucket_ind(x['Measure'], x['Value'], buckets), axis=1)

In [ ]:
labs_df.to_csv(os.path.join(DATA_DIR, 'labs.csv'), index=False)

ORDERS EVENTVAL

In [77]:
orders_df = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'))

In [79]:
orders_df['Value'] = 0
orders_df['buckets'] = 0
orders_df['eventval'] = orders_df['Measure']

In [ ]:
orders_df.to_csv(os.path.join(DATA_DIR, 'orders.csv'), index=False)

PLOT

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
N = 5

In [ ]:
counts = pd.concat([labs_count, vitals_count, orders_count, rads_count])
counts = counts.groupby('CSN').sum().reset_index().sort_values('Count')
counts = counts[counts['Count'] > N]

In [ ]:
counts['values'] = (counts['Count']/10).round().astype(int)
(counts['values'] < 40).sum() / len(counts)

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(counts['values'], bins=range(counts['values'].min(), counts['values'].max() + 2), align='left')
plt.xlabel('Events')
plt.ylabel('Frequency (Total: 116474)')
plt.grid(True, linestyle='--')
plt.show()

COUNTS

In [ ]:
visits_df = pd.read_csv(os.path.join(DATA_DIR, 'visits.csv'), usecols=['MRN', 'CSN', 'Arrival_time', 'Departure_time', 'Age', 'Gender', 'Race', 'Ethnicity'])
history_df = pd.read_csv(os.path.join(DATA_DIR, 'pmh.csv'))

In [ ]:
history_visits = visits_df[['MRN', 'CSN', 'Arrival_time', 'Departure_time']]
history_visits = history_df.merge(history_visits, on='MRN', how='left')

In [ ]:
labs_df = pd.read_csv(os.path.join(DATA_DIR, 'labs.csv'))
vitals_df = pd.read_csv(os.path.join(DATA_DIR, 'numerics.csv'))
orders_df = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'))
rads_df = pd.read_csv(os.path.join(DATA_DIR, 'rads.csv'))

In [ ]:
labs_count = labs_df.groupby('CSN')['Component_name'].count().reset_index(name='Count')
vitals_count = vitals_df.groupby('CSN')['Measure'].count().reset_index(name='Count')
orders_count = orders_df.groupby('CSN')['Procedure_name'].count().reset_index(name='Count')
rads_count = rads_df.groupby('CSN')['Study'].count().reset_index(name='Count')